# Clean up deployements

## Cleaning Up the Extended Deployment

When you have finished your experiments, you can clean up and tear down the resources. This workflow loads the final state files and systematically removes the deployed components in reverse dependency order—starting with UEs, followed by the RAN, and finally the 5G core.

In [1]:
from metamodel.models.metamodel import MetaModel
from orchestrator.models.deployment import DeploymentModel
from orchestrator.deployment import UEManagement, RANManagement, CoreManagement

metamodel = MetaModel.from_yaml_file('state/add_components/metamodel.ues.yaml')
deployment_model = DeploymentModel.from_yaml_file('state/add_components/deployment_model.ues.yaml')

ues = UEManagement(metamodel=metamodel, deployment_model=deployment_model)
await ues.remove(force=True)

ran = RANManagement(preceding=ues)
await ran.remove(force=True)

core = CoreManagement(preceding=ran)
await core.remove(force=True)


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/uninstall : Uninstall 'ueransim-sim-ue1' Helm chart] ****************

TASK [helm/uninstall : Uninstall 'ueransim-sim-ue2' Helm chart] ****************

PLAY [node-core] ***************************************************************

TASK [helm/uninstall : Uninstall 'oai-sim-ue3' Helm chart] *********************
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: The 'status.values' return value is deprecated and will be 

In a similar way, you can unprovision the kubernetes cluster and release all resources.

In [2]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

k8s = K8SManagement(preceding=core)
await k8s.remove(force=True)



PLAY [node-ran] ****************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-ues] ****************************************************************

TASK [k8s/remove : Check if kubeadm exists  on 'node-ran'] *********************

TASK [k8s/remove : Check if kubeadm exists  on 'node-ues'] *********************

TASK [k8s/remove : Check if kubeadm exists  on 'node-core'] ********************
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node-core]

TASK [k8s/remove : Reset Kubernetes node state  on 'node-core'] ****************
[WARNING]: Host 'node-ues' is using the discovered Python interpreter at '/usr/bin/pytho

In [3]:
prov = ProvisioningManagement(preceding=k8s)
await prov.unprovision(force=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/unprovision : Select project 'post5g-beta' for 'node-ran' unprovisioning] ***

TASK [provisioning/bi/unprovision : Select project 'post5g-be